# Lorinlog export (.xlsx) → Delta table

Builds `<ingest catalog>.lorinlogs.lorinlog_events` from the timestamped Excel exports
that land in the target catalog's `lorinlogs.lorinlog_files_input` volume.

**Write mode: latest file, full overwrite.** The export is a complete snapshot and
carries no primary key — nothing in it identifies a row (`Report Number` is only
half-populated and holds values like `75, 76`). So idempotency comes from replacing
the table rather than from a merge key: re-running this notebook any number of times
on the same file produces the same table. Earlier exports stay in the volume, and
earlier versions of the table stay in Delta history (`DESCRIBE HISTORY`).

### Four things this notebook is careful about

1. **The dates are text, and day-first** (`28/08/2026`). In the 2026-09-01 export,
   218 rows have day > 12 while *no* column anywhere has month > 12 — so a
   month-first parse would not fail loudly. It would null the day > 12 rows and
   **silently swap day and month** on the rest, turning `02/09/2026` into 9 February.
   Every date is parsed with an explicit `dd/MM/yyyy`, and the cell below fails if
   parsing nulls a value that was not already blank.
2. **`Impacted MW` mixes decimal separators** — `7.2` and `2,5` in the same column.
   A plain cast turns the comma form into NULL, and with ~48 rows populated nobody
   would notice. Commas are normalised before the cast, which is then also asserted.
3. **Everything is read as text.** `pandas.read_excel` infers a type per column, so
   it would coerce dates and numbers itself — differently between files, depending on
   what each file happens to contain. Reading with `dtype=str` and casting explicitly
   in Spark is what makes two runs agree.
4. **The newest file is chosen by parsed timestamp, not by name.** Filenames are
   `event-DD_MM_YYYY-HH_MM_SS.xlsx`, and sorting those as strings puts `28_08_2026`
   after `01_09_2026` — the wrong file, silently.

Set the `source_file` widget to re-ingest one specific export instead of the latest.

In [ ]:
%pip install openpyxl --quiet
dbutils.library.restartPython()

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Job parameters, with the dev values as defaults so the notebook still runs
# interactively. The job passes these so one notebook serves every target.
dbutils.widgets.text("target_catalog", "turbine_lens_dev")
dbutils.widgets.text("target_schema", "lorinlogs")
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip() or "turbine_lens_dev"
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip() or "lorinlogs"
TARGET_TABLE = "lorinlog_events"
FQN = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Derived from the SAME parameters as the target table. It was hardcoded to
# turbine_lens_dev, so the prod run read the DEV volume — caught only because the
# prod SP lacks access to it, which turned a silent cross-environment read into
# a loud PermissionError. Deriving it makes the two impossible to diverge.
VOLUME_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/lorinlog_files_input"

# Blank = ingest the newest export. Set it to re-ingest a specific file, which is
# what makes an old load reproducible after newer files have landed.
dbutils.widgets.text("source_file", "", "Source file (blank = latest)")
SOURCE_FILE_OVERRIDE = dbutils.widgets.get("source_file").strip()

# The export's columns, in order, exactly as the header row spells them. This is a
# contract, not documentation: the next cell fails if the file no longer matches, so
# a renamed or reordered column stops the run instead of quietly landing in the
# wrong Delta column.
EXPECTED_HEADER = [
    "Strap",
    "Strap status",
    "StartDateAndTime",
    "RTS Date",
    "Modification date",
    "Status",
    "Region",
    "Country",
    "Technology",
    "Asset",
    "Asset Name - Use if not in PIC",
    "Rated power",
    "Labels",
    "Title",
    "Description",
    "Report Number",
    "Components",
    "Impacted MW",
    "Category",
]

# Excel header -> Delta column. Note `StartDateAndTime` carries a date only (no time
# component in any row), so it lands as start_date rather than promising a timestamp.
COLUMN_MAP = {
    "Strap": "strap",
    "Strap status": "strap_status",
    "StartDateAndTime": "start_date",
    "RTS Date": "rts_date",
    "Modification date": "modification_date",
    "Status": "status",
    "Region": "region",
    "Country": "country",
    "Technology": "technology",
    "Asset": "asset",
    "Asset Name - Use if not in PIC": "asset_name_if_not_in_pic",
    "Rated power": "rated_power",
    "Labels": "labels",
    "Title": "title",
    "Description": "description",
    "Report Number": "report_number",
    "Components": "components",
    "Impacted MW": "impacted_mw",
    "Category": "category",
}

DATE_COLUMNS = ["start_date", "rts_date", "modification_date"]
NUMERIC_COLUMNS = ["rated_power", "impacted_mw"]
BOOLEAN_COLUMNS = ["strap"]

# Multi-value cells whose parts get sorted into a canonical order, with the separator
# the export uses. `report_number` is deliberately absent: its values are numbers as
# text ("9, 10"), so sorting them as strings would introduce a new wrongness, and no
# ordering inconsistency was observed there.
MULTIVALUE_COLUMNS = {"category": "\n", "components": "\n", "labels": ", "}

# Small enum-like columns where the source mixes case (HYDRO / Hydro).
UPPERCASE_COLUMNS = ["technology"]

print(f"target : {FQN}")
print(f"volume : {VOLUME_PATH}")
print(f"file   : {SOURCE_FILE_OVERRIDE or '(latest)'}")

In [ ]:
# ============================================================
# PICK THE SOURCE FILE
# ============================================================
import os
import re
from datetime import datetime, timezone

# event-DD_MM_YYYY-HH_MM_SS.xlsx — day first, like the dates inside the file.
FILENAME_RE = re.compile(r"^event-(\d{2})_(\d{2})_(\d{4})-(\d{2})_(\d{2})_(\d{2})\.xlsx$")


def exported_at_of(name):
    """Export timestamp from the filename, or None if it does not match."""
    m = FILENAME_RE.match(name)
    if not m:
        return None
    d, mo, y, hh, mm, ss = (int(x) for x in m.groups())
    return datetime(y, mo, d, hh, mm, ss)


# `~$…` are Excel lock files that appear whenever someone has the workbook open, and
# a leading dot is a partial upload; neither is an export.
candidates = [
    f for f in os.listdir(VOLUME_PATH)
    if f.lower().endswith(".xlsx") and not f.startswith(("~$", "."))
]
if not candidates:
    raise SystemExit(f"No .xlsx files in {VOLUME_PATH}")

unparsed = [f for f in candidates if exported_at_of(f) is None]
if unparsed:
    # Not fatal, but it means "latest" fell back to a filesystem timestamp for these,
    # which is the upload time rather than the export time.
    print(f"[warn] {len(unparsed)} file(s) do not match event-DD_MM_YYYY-HH_MM_SS.xlsx: {unparsed}")

if SOURCE_FILE_OVERRIDE:
    if SOURCE_FILE_OVERRIDE not in candidates:
        raise SystemExit(f"{SOURCE_FILE_OVERRIDE!r} is not in {VOLUME_PATH}")
    source_file = SOURCE_FILE_OVERRIDE
else:
    # Sort on the PARSED timestamp. Sorting the names as strings would rank
    # 28_08_2026 above 01_09_2026 and silently ingest the older export.
    def sort_key(f):
        parsed = exported_at_of(f)
        if parsed:
            return parsed
        return datetime.fromtimestamp(os.path.getmtime(os.path.join(VOLUME_PATH, f)))

    source_file = max(candidates, key=sort_key)

source_path = os.path.join(VOLUME_PATH, source_file)
exported_at = exported_at_of(source_file)
ingested_at = datetime.now(timezone.utc).replace(tzinfo=None)

print(f"{len(candidates)} export(s) in the volume")
print(f"ingesting   : {source_file}")
print(f"exported_at : {exported_at if exported_at else '(unknown — filename did not parse)'}")

In [ ]:
# ============================================================
# READ — everything as text, on purpose (see the header notes)
# ============================================================
import pandas as pd

pdf = pd.read_excel(
    source_path,
    sheet_name=0,
    dtype=str,            # no type inference: casting happens explicitly in Spark
    keep_default_na=False,  # no NaN either — blanks stay "" until we make them NULL
    engine="openpyxl",
)
pdf.columns = [str(c).strip() for c in pdf.columns]

print(f"{len(pdf)} data rows, {len(pdf.columns)} columns")

# The header is a contract. A renamed, added, reordered or dropped column must stop
# the run — the alternative is data landing in a column that means something else.
if list(pdf.columns) != EXPECTED_HEADER:
    missing = [c for c in EXPECTED_HEADER if c not in pdf.columns]
    added = [c for c in pdf.columns if c not in EXPECTED_HEADER]
    raise SystemExit(
        "The export's header no longer matches EXPECTED_HEADER.\n"
        f"  missing : {missing}\n"
        f"  added   : {added}\n"
        f"  found   : {list(pdf.columns)}\n"
        "Update EXPECTED_HEADER and COLUMN_MAP together, then re-run."
    )
print("header matches EXPECTED_HEADER")

# Blank counts per column, measured BEFORE any casting, so the assertions further
# down can tell "was already empty" apart from "our parse destroyed it".
blanks_before = {
    COLUMN_MAP[c]: int((pdf[c].astype(str).str.strip() == "").sum()) for c in EXPECTED_HEADER
}
row_count_in = len(pdf)

In [ ]:
# ============================================================
# CAST, CLEAN, NORMALISE
# ============================================================
from pyspark.sql import functions as F

df = spark.createDataFrame(pdf.astype(str))  # all StringType, blanks as ""
df = df.select([F.col(f"`{src}`").alias(dst) for src, dst in COLUMN_MAP.items()])

# Trim, then turn "" into NULL everywhere. The export produces both truly empty cells
# and cells holding an empty string; they mean the same thing and should read the same.
df = df.select([
    F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))).alias(c)
    for c in df.columns
])

# Case-only variants in small enum-like columns (HYDRO vs Hydro).
for c in UPPERCASE_COLUMNS:
    df = df.withColumn(c, F.upper(F.col(c)))

# Canonical ordering inside multi-value cells, so "A\nB" and "B\nA" compare equal.
for c, sep in MULTIVALUE_COLUMNS.items():
    parts = F.split(F.col(c), r"\n" if sep == "\n" else ",")
    cleaned = F.array_sort(F.array_distinct(F.filter(
        F.transform(parts, lambda x: F.trim(x)), lambda x: x != ""
    )))
    df = df.withColumn(
        c, F.when(F.col(c).isNull(), None).otherwise(F.array_join(cleaned, sep))
    )

# Keep the text each typed column came from, so the checks below can name the value
# that failed instead of guessing at it. The date and boolean copies are temporary and
# dropped before the write; the numeric ones are kept as <col>_raw, because that column
# genuinely does contain human free text (see below).
TYPED_COLUMNS = DATE_COLUMNS + NUMERIC_COLUMNS + BOOLEAN_COLUMNS
for c in TYPED_COLUMNS:
    df = df.withColumn(f"_src_{c}", F.col(c))

# Dates: explicit day-first format, and try_to_date so an unparseable value becomes
# NULL deterministically instead of depending on the session's ANSI setting. The nulls
# are then asserted against blanks_before.
for c in DATE_COLUMNS:
    df = df.withColumn(c, F.expr(f"try_to_date({c}, 'dd/MM/yyyy')"))

# Numbers. `Impacted MW` is maintained by hand and holds more than numbers — the
# 2026-09-01 export contains '169.1 MW' and '30.6 + 17.8 = 48.4'. So before casting:
# take what follows the last '=' (someone writing a sum means the total), drop a unit
# suffix, remove spaces including non-breaking ones, and turn the comma decimal
# separator into a point ('2,5' would otherwise cast to NULL, unnoticed among ~48
# populated rows). Anything still unparseable stays readable in <col>_raw and is
# reported by the next cell — never guessed at.
for c in NUMERIC_COLUMNS:
    cleaned = F.col(c)
    cleaned = F.when(
        cleaned.contains("="), F.element_at(F.split(cleaned, "="), -1)
    ).otherwise(cleaned)
    cleaned = F.regexp_replace(cleaned, r"(?i)\s*(gwh|mwh|mwc|mw|kwh|kw)\s*$", "")
    cleaned = F.regexp_replace(cleaned, r"[\s\u00a0]", "")
    cleaned = F.regexp_replace(cleaned, ",", ".")
    df = df.withColumn("_cleaned", cleaned)
    df = df.withColumn(c, F.expr("try_cast(_cleaned AS DOUBLE)")).drop("_cleaned")
    df = df.withColumnRenamed(f"_src_{c}", f"{c}_raw")

for c in BOOLEAN_COLUMNS:
    df = df.withColumn(c, F.expr(f"try_cast({c} AS BOOLEAN)"))

# Description is HTML. Keep it verbatim and add a readable column beside it; block
# tags become newlines first so sentences do not run together, then tags go, then the
# handful of entities this source actually produces.
text = F.col("description")
text = F.regexp_replace(text, r"(?i)<br\s*/?>|</p>|</div>|</li>|</tr>", "\n")
text = F.regexp_replace(text, r"<[^>]*>", "")
for entity, repl in [
    ("&nbsp;", " "), ("&amp;", "&"), ("&lt;", "<"), ("&gt;", ">"),
    ("&quot;", '"'), ("&#39;", "'"), ("&rsquo;", "'"), ("&eacute;", "e"),
]:
    text = F.regexp_replace(text, entity, repl)
text = F.regexp_replace(text, r"[ \t]+", " ")
text = F.regexp_replace(text, r"\n{3,}", "\n\n")
df = df.withColumn(
    "description_text", F.when(F.col("description").isNull(), None).otherwise(F.trim(text))
)

# Provenance: which export this row came from, when that export was produced, and
# when it was loaded. The table is overwritten per run, so these answer "what am I
# looking at" without going back to the volume.
df = (
    df.withColumn("source_file", F.lit(source_file))
      .withColumn("exported_at", F.lit(exported_at).cast("timestamp"))
      .withColumn("ingested_at", F.lit(ingested_at).cast("timestamp"))
)

# Deliberately not cached: serverless rejects PERSIST/CACHE
# ([NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE), and at a few hundred rows the
# recomputation the checks below trigger costs nothing.
print(f"{df.count()} rows shaped, {len(df.columns)} columns")
df.printSchema()

In [ ]:
# ============================================================
# DATA QUALITY — the casts must not have destroyed anything
# ============================================================
problems = []

n_out = df.count()
if n_out != row_count_in:
    problems.append(f"row count changed: read {row_count_in}, shaped {n_out}")

# HARD: a date or boolean that was present in the file but is NULL after casting means
# the format assumption is wrong, and this is the check that would catch a month-first
# date format. Dates in this export are machine-written, so there is no excuse for one
# not to parse — it fails the run.
for c in DATE_COLUMNS + BOOLEAN_COLUMNS:
    src = F.col(f"_src_{c}")
    lost = df.filter(F.col(c).isNull() & src.isNotNull())
    n_lost = lost.count()
    if n_lost:
        samples = [r[0] for r in lost.select(src).limit(5).collect()]
        problems.append(f"{c}: {n_lost} value(s) present in the file failed to parse: {samples}")
    else:
        print(f"  ok  {c:26} {df.filter(F.col(c).isNull()).count():>4} null "
              f"/ {blanks_before[c]:>4} blank in file")

# SOFT: the numeric columns are typed by hand upstream and contain prose ('169.1 MW',
# '30.6 + 17.8 = 48.4'). Failing the load over that would make the notebook unusable,
# and nulling it silently is the thing being avoided — so it is reported every run and
# the text is preserved in <col>_raw. What IS asserted is that nothing was lost.
for c in NUMERIC_COLUMNS:
    raw = F.col(f"{c}_raw")
    unparsed = df.filter(F.col(c).isNull() & raw.isNotNull())
    n_unparsed = unparsed.count()
    print(f"  ok  {c:26} {df.filter(F.col(c).isNull()).count():>4} null "
          f"/ {blanks_before[c]:>4} blank in file")
    if n_unparsed:
        vals = [r[0] for r in unparsed.select(raw).distinct().collect()]
        print(f"      [dq] {n_unparsed} row(s) not numeric, kept in {c}_raw: {vals}")
    if df.filter(F.col(c).isNotNull() & raw.isNull()).count():
        problems.append(f"{c}: parsed a value whose raw text is missing — bug in the cast")

if problems:
    raise SystemExit("Data quality check failed:\n  - " + "\n  - ".join(problems))
print("\nno values lost to casting")

# What normalising changed, and what is still inconsistent upstream. Printed rather
# than fixed silently, so the source system's oddities stay visible.
print("\n[dq] value spread after normalising")
for c in ["status", "region", "technology", "labels", "category"]:
    vals = [r[c] for r in df.select(c).distinct().collect() if r[c] is not None]
    print(f"  {c:12} {len(vals):>3} distinct  {sorted(vals)[:6]}")

print("\n[dq] date range")
df.select(
    F.min("start_date").alias("earliest_start"),
    F.max("start_date").alias("latest_start"),
    F.max("modification_date").alias("latest_modification"),
).show(truncate=False)

In [ ]:
# ============================================================
# WRITE — full overwrite of the snapshot
# ============================================================
# The date/boolean text copies existed only for the checks above; <col>_raw for the
# numeric columns is kept on purpose.
df_out = df.drop(*[f"_src_{c}" for c in DATE_COLUMNS + BOOLEAN_COLUMNS])

(
    df_out.write
      .mode("overwrite")
      # The export's shape can change between files; a schema change should land in
      # the table rather than fail after the header check has already approved it.
      .option("overwriteSchema", "true")
      .saveAsTable(FQN)
)
print(f"wrote {df_out.count()} rows to {FQN}")

spark.sql(f"""
    COMMENT ON TABLE {FQN} IS
    'Lorinlog event export, loaded from the .xlsx files in {VOLUME_PATH} by
     data/jobs/lorinlog_excel_to_delta.ipynb. Contains ONE export at a time: each run
     overwrites the table with the newest file, because the export is a full snapshot
     and carries no primary key. See source_file / exported_at for which snapshot this
     is, DESCRIBE HISTORY for previous ones, and the volume for the raw files.'
""")

In [ ]:
# ============================================================
# VERIFY
# ============================================================
display(spark.sql(f"""
    SELECT source_file, exported_at, ingested_at, COUNT(*) AS rows,
           MIN(start_date) AS earliest_start, MAX(start_date) AS latest_start,
           COUNT(rated_power) AS rated_power_filled,
           COUNT(impacted_mw) AS impacted_mw_filled
    FROM {FQN}
    GROUP BY ALL
"""))

In [ ]:
display(spark.sql(f"""
    SELECT start_date, status, region, country, technology, asset,
           rated_power, impacted_mw, category, title,
           left(description_text, 120) AS description_text_head
    FROM {FQN}
    ORDER BY start_date DESC
    LIMIT 10
"""))

In [ ]:
# ============================================================
# GENIE VIEW
# ============================================================
# What the `lorin_qualitative_context` Genie space actually reads — it references this
# view eleven times and never the base table. Until now it existed ONLY in the
# workspace: created by hand, in no file anywhere, so a fresh environment got the table
# and a Genie space pointing at nothing.
#
# Deliberately narrower than the base table: business columns only, and
# `description_text` surfaced as `description` because that is the name the space's
# instructions use. Do not point the space at `lorinlog_events` expecting this shape.
#
# CREATE OR REPLACE, not CREATE: this job runs on every file arrival, so the statement
# has to be idempotent. It is cheap — a view stores no data — and it makes the view
# self-healing rather than something to remember after a redeploy.
spark.sql(f"""
CREATE OR REPLACE VIEW {TARGET_CATALOG}.{TARGET_SCHEMA}.lorinlog_events_genie AS
SELECT
  title,
  description_text AS description,
  labels,
  report_number,
  asset,
  technology,
  rated_power,
  components,
  region,
  country,
  status,
  category,
  impacted_mw,
  start_date,
  rts_date,
  modification_date
FROM {TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}
""")
print(f"view ready: {TARGET_CATALOG}.{TARGET_SCHEMA}.lorinlog_events_genie")
